# 단일 어댑터 CLV 효과 식별 실험

이 노트북은 M2 임베딩의 `single_full`이 사용자별 CLV 관련 행동정보에서 실제 이득을 얻는지 검증합니다. seed 42의 validation만 사용하며 test와 holdout은 만들지 않습니다.


In [ ]:
from google.colab import drive

import torch

assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
print('GPU:', torch.cuda.get_device_name(0))

drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
REPO_DIR = Path('/content/clv-m2-lightgcn-runner')
REVIEWED_SHA = '94fbc50de735bf88dffaf3651196be2b48aa82d1'

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REVIEWED_SHA], check=True)
actual_sha = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True,
).stdout.strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
os.chdir(REPO_DIR)
print('검토된 소스:', actual_sha)


In [ ]:
import json
from pathlib import Path

from lightgcn_clv_single import (
    configure_single_run,
    preflight_summary,
    run_experiment,
)

DATASET = 'dunnhumby'  # 'dunnhumby' 또는 'hm'
cfg = configure_single_run(
    DATASET,
    seed_list=(42,),
    eval_test=False,
    eval_holdout=False,
    out_dir=f'/content/drive/MyDrive/논문/data/results_clv_single_{DATASET}',
    m1_checkpoint_dir=f'/content/drive/MyDrive/논문/data/results_v3_{DATASET}',
)
# 과거 single_full을 재사용하려면 검토 당시 별도 보관한 JSON 경로와 체크포인트 SHA-256을 둘 다 입력하세요.
# 해시가 없으면 안전하게 single_full을 새로 학습합니다.
reuse_full_result_json = None
reuse_full_checkpoint_sha256 = None


In [ ]:
summary = preflight_summary(cfg)
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('\n아직 데이터 전처리나 모델 학습은 시작되지 않았습니다.')
print('기존 Dunnhumby 결과 재사용:', reuse_full_result_json)
print('신뢰된 체크포인트 SHA-256:', reuse_full_checkpoint_sha256)


## 고비용 실행 승인

위 설정 전체를 검토한 뒤에만 아래 값을 `True`로 바꾸세요. 이 셀 전까지 모델 학습은 시작되지 않습니다.


In [ ]:
ACKNOWLEDGE_HIGH_COST = False
assert ACKNOWLEDGE_HIGH_COST, '설정 검토 후 ACKNOWLEDGE_HIGH_COST=True로 바꾸세요.'
result_df = run_experiment(
    cfg,
    reuse_full_result_json=reuse_full_result_json,
    reuse_full_checkpoint_sha256=reuse_full_checkpoint_sha256,
)


In [ ]:
from IPython.display import display

display(result_df.sort_values(['model_id', 'split', 'lambda']))
print('선택 λ:', result_df.attrs['selected_lambda'])
print('λ 선택조건 통과:', result_df.attrs['lambda_selection_success'])
decision = result_df.attrs['screening_decision']
print('최종 성공:', decision['success'])
print('판정 이유:', decision['reason'])
print('실패 대조군:', decision['failed_controls'])
print('아이템 정보 제거 비교:', decision['mechanism_comparison'])
print('결과 파일:')
for label, path in result_df.attrs['result_paths'].items():
    print(f' - {label}: {path}')
